In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(1 , 32 , kernel_size = 3) # (28x28) -> (26x26)
        self.conv2 = nn.Conv2d(32 , 64 , kernel_size = 3) # (26x26) -> (24x24)

        self.pool = nn.MaxPool2d(2 , 2) # resizeing 2 time

        self.fc1 = nn.Linear(64 * 5 * 5 , 128)
        self.fc2 = nn.Linear(128 , 10) # 10 classes

    def forward(self , x):
        x = self.pool(F.relu(self.conv1(x))) # -> (32 , 13 , 13)
        x = self.pool(F.relu(self.conv2(x))) # -> (64 , 6 , 6)

        x = x.view(x.size(0), -1)

        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x

Uploading data

In [3]:
from torchvision import datasets , transforms
from torch.utils.data import DataLoader

transform = transforms.ToTensor()

train_data = datasets.MNIST(root = "./data" , train = True , transform = transform , download = True)
test_data = datasets.MNIST(root = "./data" , train = False , transform = transform)

train_loader = DataLoader(train_data , batch_size = 64 ,  shuffle = True)
test_loader = DataLoader(test_data , batch_size = 64)

Training

In [4]:
model = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters() , lr = 0.001)

for epoch in range(5):
    total_loss = 0

    for images , labels in train_loader:
        outputs = model(images)
        loss = criterion(outputs , labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    print("Torch: " , epoch + 1 ,"Loss: " , total_loss)

Torch:  1 Loss:  178.03463197988458
Torch:  2 Loss:  52.307726148865186
Torch:  3 Loss:  35.240644357632846
Torch:  4 Loss:  26.85735506529454
Torch:  5 Loss:  20.20950694817293


Accuracy rating

In [6]:
correct = 0
total = 0

model.eval()

with torch.no_grad():
    for images , labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs , 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()
accuracy = 100 * correct / total
print("Accuracy: " , accuracy , "%")

Accuracy:  98.92 %
